# DriveLegal AI - RAG Pipeline (FAISS & HuggingFace)

This notebook demonstrates the complete Retrieval-Augmented Generation (RAG) pipeline for **DriveLegal AI**.
It walks through loading the pre-processed traffic violation data, generating vector embeddings using a local HuggingFace model (`all-MiniLM-L6-v2`), saving the FAISS index (`.pkl`), and finally querying the LLM.

## 1. Setup Environment

In [9]:
# Install dependencies if not already installed (uncomment to run)
# %pip install faiss-cpu langchain-community sentence-transformers google-generativeai

In [10]:
import os
import sys
import json

# Add backend to Python path to allow importing our custom modules
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(BASE_DIR, "backend"))

print(f"Project Base Directory: {BASE_DIR}")

Project Base Directory: d:\DriveLegal


## 2. Load Processed Data
We load the `chunks.json` which contains the text snippets of traffic violations, fines, and sections.

In [11]:
from langchain_core.documents import Document

chunks_path = os.path.join(BASE_DIR, "data", "processed", "chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

docs = [
    Document(
        page_content=item["text"], 
        metadata={"chunk_id": item["chunk_id"], "section": item.get("section", "")},
        id=item["chunk_id"]
    ) 
    for item in chunks_data
]

print(f"Loaded {len(docs)} document chunks.")
print("\nSample Document:\n", docs[0].page_content)

Loaded 298 document chunks.

Sample Document:
 === BankBazaar Traffic Fines India ===
Source: https://www.bankbazaar.com/driving-licence/traffic-fines.html


## 3. Train and Save FAISS Index (.pkl)
We embed the documents using `all-MiniLM-L6-v2`. FAISS calculates the vector similarities and saves the model as `index.faiss` and `index.pkl` in the `embeddings/` folder.

In [12]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Initialize embedding model
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Build FAISS vector store
vectorstore = FAISS.from_documents(docs, embedder)

# 3. Save locally as .pkl and .faiss
embeddings_dir = os.path.join(BASE_DIR, "embeddings")
os.makedirs(embeddings_dir, exist_ok=True)

vectorstore.save_local(embeddings_dir)
print(f"\u2705 FAISS index successfully trained and saved to: {embeddings_dir}")

C:\Users\SUBHANKAR\AppData\Local\Temp\ipykernel_33652\1963614867.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
C:\Users\SUBHANKAR\AppData\Local\Temp\ipykernel_33652\1963614867.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19645.01it/s]


✅ FAISS index successfully trained and saved to: d:\DriveLegal\embeddings


## 4. Querying the Retriever
We load the saved `.pkl` file and perform a similarity search to find the most relevant traffic laws.

In [13]:
# Load from the saved .pkl file
loaded_vectorstore = FAISS.load_local(
    embeddings_dir, 
    embedder, 
    allow_dangerous_deserialization=True
)

query = "What is the penalty for driving without a helmet?"
results = loaded_vectorstore.similarity_search(query, k=3)

print(f"\n\U0001f50d Query: '{query}'\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:250] + "...")
    print()


🔍 Query: 'What is the penalty for driving without a helmet?'

--- Result 1 ---
[SECTION 194D] - Riding without Helmet
No person shall drive or cause or allow to be driven, in any public place, a motor cycle without wearing protective headgear conforming to the standards of Bureau of Indian Standards:
Provided that this section ...

--- Result 2 ---
[SECTION 194B(1)] - Driving without Seatbelt
Whoever drives a motor vehicle without wearing a safety belt or carries passengers not wearing safety belts shall be punishable with a fine of one thousand rupees (₹1,000)....

--- Result 3 ---
Traffic Violations: Driving without Helmet (Rider/Pillion Rider) | Fine in Delhi: Rs.1,000 | Fine in Maharashtra: Rs.500, plus disqualification for holding of driving licence for three months | Fine in Tamil Nadu: Rs.1,000 | Fine in Uttar Pradesh: Rs...



## 5. End-to-End Test with Chatbot Pipeline
Finally, we can test our backend integration natively to ensure the LLM generates a response based on the retrieved context.

In [15]:
from ingestion.pipeline import RAGPipeline
import asyncio

async def test_full_rag():
    # Initialize the backend pipeline
    pipeline = RAGPipeline()
    
    test_query = "I got caught driving without a helmet in Delhi. How much is the fine?"
    print(f"User: {test_query}\n")
    
    # The pipeline internally embeds the query, searches FAISS, and queries the LLM
    response = await pipeline.run(test_query, state="Delhi")
    
    print("\U0001f916 DriveLegal AI:")
    print(response["answer"])

# Run the async test
await test_full_rag()

User: I got caught driving without a helmet in Delhi. How much is the fine?

🤖 DriveLegal AI:
In Delhi, driving a motorcycle without a helmet is punishable with a fine of **₹1,000** and a **3-month license suspension** under **Section 194D** of the Motor Vehicles Act.
